# 38 — The deployment combination: drift gate → drift-aware posterior (System I, N3, D1 only)

**Question.** Notebook 09 gives a posterior that remedies the Sc7 sensor drift at a price (wider β, macro-F1 −0.03); notebook 11b gives a detector that flags the drift without any posterior. What does a deployment that
uses the detector as a gate achieve? **Rule (user decision, fixed before running; runbook §2f row g2):** a window is *flagged* iff it is unsaturated (at most 50% of its samples at Qc ≥ 395 L/min) and |z_T| > 10, with
z_T = (mean measured T − T_sp)/(0.2 K/√120); **flagged windows use the drift-aware posterior (notebook 09 ensemble), unflagged windows the production 2-D posterior (notebook 36 ensemble).** The production model stays the 2-D posterior.
**Data: D1 only** (50 windows per scenario; the threshold 10 was chosen on L1). Per scenario Sc1–Sc5 and Sc7: class accuracy, β 90% coverage, β sd, β bias; the gate's flag rate (false-alarm rate in Sc1–Sc5, detection rate in Sc7) for all windows and for
unsaturated windows; the Sc7 split by saturation; the production-only and drift-aware-only posteriors on the same windows for reference; and a **labelled sensitivity** without the saturation guard (|z_T| > 10 on every window). Macro-F1 over Sc1–Sc5.
**Checks:** the gate's detection of Sc7 in unsaturated windows is 1.000 as in notebook 11b; everything finite; samples inside the prior. No decision rule. With 50 windows per scenario, coverage has a standard error of about 0.04.


In [1]:
import json, os, sys, time
from pathlib import Path
import numpy as np, pandas as pd
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "system_I" and NB_DIR.parent.name == "notebooks", f"run from notebooks/system_I (cwd {NB_DIR})"
ROOT = NB_DIR.parent.parent
for p in (ROOT / "src", ROOT / "scripts"):
    sys.path.insert(0, str(p))
os.environ["CSTR_SENSOR_MODEL"] = "N3"
import importlib, ws_eval, ws_metrics, ws_phase3, nb09_drift_aware
for _m in (ws_eval, ws_metrics, ws_phase3, nb09_drift_aware):
    importlib.reload(_m)
we, wm, P3, D = ws_eval, ws_metrics, ws_phase3, nb09_drift_aware
R = ROOT / "results"; T0 = time.time()
SCN = {1: "Sc1", 2: "Sc2", 3: "Sc3", 4: "Sc4", 5: "Sc5", 7: "Sc7"}
d1 = P3.load_d1(); X = P3.production_features(d1)
post_prod, _ = P3.load_production_posterior(); post_da = D.ensemble()
s_prod, _ = we.sample_within_prior(post_prod, X, 1000, rng_seed=21)            # (300, 1000, 2)
s_da3, _ = D.sample_box(post_da, X, 1000, seed=22); s_da = s_da3[:, :, :2]
assert np.isfinite(s_prod).all() and np.isfinite(s_da).all()
sid = d1["scenario_id"]; th = d1["theta"][:, 2:].astype(float)
z = (d1["x"][:, :, 1].mean(1) - 312.5) / (0.2 / np.sqrt(120.0)); sat = (d1["x"][:, :, 3] >= 395).mean(1) > 0.5
flag = (~sat) & (np.abs(z) > 10.0); flag_ng = np.abs(z) > 10.0
print(f"D1: {len(X)} windows | flagged (guarded gate): {flag.sum()} | flagged without the guard: {flag_ng.sum()} | saturated windows: {sat.sum()}")


D1: 300 windows | flagged (guarded gate): 34 | flagged without the guard: 145 | saturated windows: 112


## 1. The gate: false-alarm and detection rates

In [2]:
rows = []
for sc, nm in SCN.items():
    m = sid == sc; u = m & ~sat
    rows.append(dict(scenario=nm, n=int(m.sum()), n_unsaturated=int(u.sum()), flag_rate_all_windows=float(flag[m].mean()), flag_rate_unsaturated=float(flag[u].mean()) if u.any() else np.nan,
                     flag_rate_no_guard_all=float(flag_ng[m].mean()), role="detection rate (drift present)" if sc == 7 else "false-alarm rate"))
gate = pd.DataFrame(rows); display(gate.round(3)); gate.to_csv(R / "38_N3_gate_rates.csv", index=False)
det_u = float(flag[(sid == 7) & ~sat].mean()); assert det_u == 1.0, det_u
print("check passed: Sc7 detection in unsaturated windows = 1.000 (as notebook 11b); Sc7 saturated windows:", int(((sid == 7) & sat).sum()), "(never flagged by the guarded gate, by construction)")


,scenario,n,n_unsaturated,flag_rate_all_windows,flag_rate_unsaturated,flag_rate_no_guard_all,role
0,Sc1,50,35,0.02,0.029,0.32,false-alarm rate
1,Sc2,50,29,0.00,0.000,0.42,false-alarm rate
2,Sc3,50,46,0.02,0.022,0.10,false-alarm rate
3,Sc4,50,34,0.00,0.000,0.30,false-alarm rate
4,Sc5,50,13,0.02,0.077,0.76,false-alarm rate
5,Sc7,50,31,0.62,1.000,1.00,detection rate (drift present)


check passed: Sc7 detection in unsaturated windows = 1.000 (as notebook 11b); Sc7 saturated windows: 19 (never flagged by the guarded gate, by construction)


## 2. Per-scenario numbers for the combination (headline), with the references

In [3]:
def combo(f):
    return np.where(f[:, None, None], s_da, s_prod)
def table(s, label):
    yt = wm.class_of_theta(th); yp = wm.class_of_samples(s); q05 = np.quantile(s, .05, axis=1); q95 = np.quantile(s, .95, axis=1); mean = s.mean(1); sd = s.std(1)
    out = []
    for sc, nm in SCN.items():
        m = sid == sc
        out.append(dict(variant=label, scenario=nm, n=int(m.sum()), class_acc=float((yt[m] == yp[m]).mean()), cov90_b=float(((th[m, 1] >= q05[m, 1]) & (th[m, 1] <= q95[m, 1])).mean()),
                        sd_b=float(sd[m, 1].mean()), bias_b=float((mean[m, 1] - th[m, 1]).mean())))
    return pd.DataFrame(out), yt, yp
tabs, preds = {}, {}
for label, s in (("gate + drift-aware (guarded; headline)", combo(flag)), ("production 2-D only", s_prod), ("drift-aware only", s_da), ("sensitivity: gate without the saturation guard", combo(flag_ng))):
    tabs[label], yt, preds[label] = table(s, label)
allt = pd.concat(tabs.values(), ignore_index=True); allt["scenario"] = pd.Categorical(allt.scenario, list(SCN.values()))
display(allt.sort_values(["scenario", "variant"]).round(3).reset_index(drop=True)); allt.to_csv(R / "38_N3_D1_three_numbers.csv", index=False)
mf = []
for label, yp in preds.items():
    m15 = np.isin(sid, [1, 2, 3, 4, 5]); mf.append(dict(variant=label, macro_f1_Sc1_5=wm.macro_f1(yt[m15], yp[m15]), macro_f1_all_six=wm.macro_f1(yt, yp), accuracy_Sc1_5=float((yt[m15] == yp[m15]).mean())))
mf = pd.DataFrame(mf); display(mf.round(3)); mf.to_csv(R / "38_N3_macro_f1.csv", index=False)


,variant,scenario,n,class_acc,cov90_b,sd_b,bias_b
0,drift-aware only,Sc1,50,0.86,0.94,0.067,-0.034
1,gate + drift-aware (guarded; headline),Sc1,50,0.98,0.94,0.046,0.001
2,production 2-D only,Sc1,50,0.98,0.94,0.046,0.001
3,sensitivity: gate without the saturation guard,Sc1,50,0.86,0.96,0.069,-0.032
4,drift-aware only,Sc2,50,0.86,0.90,0.079,-0.019
5,gate + drift-aware (guarded; headline),Sc2,50,0.84,0.92,0.047,0.007
6,production 2-D only,Sc2,50,0.84,0.92,0.047,0.007
7,sensitivity: gate without the saturation guard,Sc2,50,0.84,0.92,0.080,-0.020
8,drift-aware only,Sc3,50,0.96,0.90,0.044,-0.009
9,gate + drift-aware (guarded; headline),Sc3,50,0.98,0.88,0.042,-0.006


,variant,macro_f1_Sc1_5,macro_f1_all_six,accuracy_Sc1_5
0,gate + drift-aware (guarded; headline),0.939,0.901,0.940
1,production 2-D only,0.939,0.874,0.940
2,drift-aware only,0.896,0.883,0.900
3,sensitivity: gate without the saturation guard,0.894,0.882,0.896


## 3. Sc7 split by saturation (the guard exempts saturated windows)

In [4]:
s_c = combo(flag); rows = []
for lab, m in (("Sc7 unsaturated (flagged -> drift-aware)", (sid == 7) & ~sat), ("Sc7 saturated (unflagged -> production)", (sid == 7) & sat)):
    s = s_c[m]; q05 = np.quantile(s, .05, axis=1)[:, 1]; q95 = np.quantile(s, .95, axis=1)[:, 1]
    rows.append(dict(group=lab, n=int(m.sum()), cov90_b=float(((th[m, 1] >= q05) & (th[m, 1] <= q95)).mean()) if m.any() else np.nan,
                     bias_b=float((s.mean(1)[:, 1] - th[m, 1]).mean()) if m.any() else np.nan, sd_b=float(s.std(1)[:, 1].mean()) if m.any() else np.nan))
sp = pd.DataFrame(rows); display(sp.round(3)); sp.to_csv(R / "38_N3_sc7_by_saturation.csv", index=False)
print(f"notebook wall time {(time.time() - T0) / 60:.1f} min")


,group,n,cov90_b,bias_b,sd_b
0,Sc7 unsaturated (flagged -> drift-aware),31,0.839,-0.002,0.034
1,Sc7 saturated (unflagged -> production),19,0.000,-0.398,0.031


notebook wall time 0.2 min


## 6. Interpretation

All numbers are D1, 50 windows per scenario (coverage SE about 0.04, class accuracy about 0.05); the threshold 10 was chosen on L1, so D1 is the honest test of it.

**The gate.** With the guarded rule (unsaturated and |z_T| > 10) the false-alarm rate is 0–2% of all windows per scenario (Sc1, Sc3, Sc5: one window each; Sc2, Sc4: none), 3 of 250 pooled (1.2%), or 3 of 157 unsaturated windows (1.9%). The detection rate in Sc7 is **1.00 in the unsaturated windows (31 of 31)**
and **0.62 over all Sc7 windows**, because the guard never flags a saturated window and 19 of the 50 Sc7 windows are saturated.

**The combination keeps the well-specified performance.** In Sc1–Sc5 it equals the production posterior within one window (macro-F1 0.939 for both; β coverage 0.88–0.98, β sd 0.042–0.047 in every scenario), because the gate almost never fires there. The drift-aware posterior alone would cost macro-F1 (0.896) and widen β (sd 0.061–0.079 in Sc1, Sc2, Sc4, Sc5):
the price measured in notebook 09 on L1 shows up here too.

**Sc7: the unsaturated windows are repaired, the saturated windows are not.** The 31 flagged windows go to the drift-aware posterior and are right (β bias −0.002, coverage 0.84, sd 0.034). The 19 saturated windows are unflagged by construction, go to the production posterior and are as wrong as before (β bias −0.398, coverage 0.000, sd 0.031):
the silent failure persists in exactly the windows where the loop has lost control and the z_T statistic is uninformative. Over all Sc7 windows the combination gives class accuracy 0.72 (production alone 0.52), β coverage 0.52 (0.00) and β bias −0.153 (−0.394), i.e. **it roughly halves the failure rather than removing it**.

**The unguarded variant (sensitivity, not the pre-set rule).** Flagging on |z_T| > 10 in every window, saturated or not, routes all saturated windows to the drift-aware posterior. Sc7 is then fully repaired (accuracy 0.86, coverage 0.88, bias −0.017), the false-alarm rate is 10–76% (saturated windows are routinely flagged because the temperature leaves the setpoint),
and the well-specified scenarios pay the drift-aware price (macro-F1 0.894 against 0.939 on D1; β sd 0.046–0.080; Sc5, the most saturated scenario, widens to 0.079). It is equivalent to "use the drift-aware posterior whenever the valve is saturated or an offset is flagged".

**Operational message.** (1) The guarded gate is cheap and almost free of false alarms, and it repairs every unsaturated drifted window. (2) A saturated window needs its own rule: either treat it as *unusable by the production posterior* (the loop has lost control, the sensor and the process cannot be told apart by z_T)
and report it as such, or accept the price of the drift-aware posterior there. This is a decision for the practitioner, not one the data settle: it trades a β sd of about 0.08 against a silent bias of about 0.4 in a regime that is 8–74% of the D1 windows depending on the scenario. (3) The 62%/38% split is specific to the Sc7 operating point (valve saturation at β = 0.85 with the inlet varying over ±2 K).

**Limits.** D1 only, 50 windows per scenario, one drift size (+2 K), one threshold; the false-alarm estimates are 0–1 windows per scenario; the saturated Sc7 result is 19 windows.